# Building AI Agents with Persistent Memory using Microsoft Agent Framework and Azure AI Search (C#)

This notebook demonstrates how to build an intelligent travel booking agent that remembers user preferences across conversations. By combining the Microsoft Agent Framework and Azure AI Search, we create an agent that provides personalized travel recommendations based on historical interactions.

## What You'll Learn:
1. **Agent Memory**: How to implement persistent memory for AI agents
2. **Azure AI Search as Data Store**: Store and retrieve hotel data and user memories
3. **Persistent User Preferences**: Remember user preferences across different chat sessions
4. **Agent Tools**: Build custom tools that leverage both memory and search capabilities
5. **Microsoft Agent Framework**: Use the modern .NET agent framework for building intelligent agents

## Prerequisites:
- Azure AI Foundry or Azure OpenAI deployment configured
- Azure AI Search service created
- Understanding of basic AI agent concepts

## Understanding the Memory Architecture

### Memory in AI Agents:

**Agent Memory** enables:
- **Long-term Memory**: Store user preferences, past interactions, and learned information
- **Contextual Retrieval**: Retrieve relevant memories based on current conversation
- **User-specific Storage**: Maintain separate memory spaces for different users
- **Personalization**: Provide customized responses based on stored preferences

### How the Components Work Together:
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Agent          │────▶│   Memory Store   │────▶│  Azure AI       │
│  Framework      │     │   (Simple Dict)  │     │  Search         │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                       │                         │
         │                       │                         │
    Processes              Stores/Retrieves          Vector Store
    User Input             User Preferences         for Hotel Data
                          & Context                  
```

In [1]:
Console.WriteLine("Hello, World!");

In [2]:
// 📦 Install Required NuGet Packages
#r "nuget: Newtonsoft.Json"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.1.1-preview.1.25612.2"
//#r "nuget: Microsoft.Extensions.AI, 10.*"
//#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.*-*"
#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.260108.1"
//#r "nuget: Microsoft.Agents.AI.OpenAI, 1.*-*"
#r "nuget: Azure.AI.OpenAI, 2.8.0-beta.1"
//#r "nuget: Azure.AI.OpenAI, *-*"
#r "nuget: Azure.Identity"
#r "nuget: Azure.Search.Documents"
#r "nuget: DotNetEnv"

Installed Packages Azure.AI.OpenAI, 2.8.0-beta.1 Azure.Identity, 1.17.1 Azure.Search.Documents, 11.7.0 DotNetEnv, 3.1.1 Microsoft.Agents.AI.OpenAI, 1.0.0-preview.260108.1 Microsoft.Extensions.AI.OpenAI, 10.1.1-preview.1.25612.2 Newtonsoft.Json, 13.0.4

## Import Required Packages

In [3]:
using System;
using System.Collections.Generic;
using System.ComponentModel;
using System.Linq;
using System.Net.Http;
using System.Threading.Tasks;
using Newtonsoft.Json;

// Azure Search
using Azure;
using Azure.Search.Documents;
using Azure.Search.Documents.Indexes;
using Azure.Search.Documents.Indexes.Models;
using Azure.Search.Documents.Models;

// Azure OpenAI and Agent Framework
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;

## Environment Configuration

In [4]:
// Load environment variables from .env file
DotNetEnv.Env.Load();
Console.WriteLine("✅ Environment variables loaded from .env file");

In [5]:
// Azure OpenAI Configuration
var azureOpenAIDeployment = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_MODEL");
var azureOpenAIEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_ENDPOINT");
var apiVersion = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_VERSION");

// Azure AI Search Configuration
var searchServiceEndpoint = Environment.GetEnvironmentVariable("AZURE_SEARCH_SERVICE_ENDPOINT");
var searchApiKey = Environment.GetEnvironmentVariable("AZURE_SEARCH_API_KEY");

// Index names
var travelIndexName = "travel-hotels";
var memoryIndexName = "mem0-memories";

// Print environment variables
Console.WriteLine("Environment Variables:");
Console.WriteLine("=" + new string('=', 49));
Console.WriteLine($"AZURE_AI_FOUNDRY_MODEL: {azureOpenAIDeployment}");
Console.WriteLine($"AZURE_AI_FOUNDRY_ENDPOINT: {azureOpenAIEndpoint}");
Console.WriteLine($"AZURE_OPENAI_API_VERSION: {apiVersion}");
Console.WriteLine($"AZURE_SEARCH_SERVICE_ENDPOINT: {searchServiceEndpoint}");
Console.WriteLine($"AZURE_SEARCH_API_KEY: {(searchApiKey != null ? "****" : null)}");
Console.WriteLine("=" + new string('=', 49));

AZURE_AI_FOUNDRY_MODEL: gpt-5-mini
AZURE_AI_FOUNDRY_ENDPOINT: https://myresearchfoundry.openai.azure.com/
AZURE_OPENAI_API_VERSION: 2024-02-01
AZURE_SEARCH_SERVICE_ENDPOINT: https://ibecsearch.search.windows.net
AZURE_SEARCH_API_KEY: ****


## Initialize Azure AI Search for Travel Data

First, we'll set up Azure AI Search with sample hotel and destination data that our agent can search through.

In [6]:
// Initialize search clients
var credential = new AzureKeyCredential(searchApiKey);
var indexClient = new SearchIndexClient(new Uri(searchServiceEndpoint), credential);

// Create travel data index if it doesn't exist
var travelFields = new List<SearchField>
{
    new SimpleField("id", SearchFieldDataType.String) { IsKey = true },
    new SearchableField("name"),
    new SearchableField("description"),
    new SearchableField("location"),
    new SearchableField("amenities"),
    new SimpleField("price_per_night", SearchFieldDataType.Double),
    new SimpleField("rating", SearchFieldDataType.Double),
    new SearchableField("tags") { IsFilterable = true }
};

var travelIndex = new SearchIndex(travelIndexName, travelFields);

try
{
    await indexClient.GetIndexAsync(travelIndexName);
    Console.WriteLine($"✅ Index '{travelIndexName}' already exists");
}
catch
{
    await indexClient.CreateIndexAsync(travelIndex);
    Console.WriteLine($"✅ Created index '{travelIndexName}'");
}

// Initialize search client for travel data
var travelSearchClient = new SearchClient(new Uri(searchServiceEndpoint), travelIndexName, credential);

In [7]:
// Define hotel class
public class Hotel
{
    public string id { get; set; }
    public string name { get; set; }
    public string description { get; set; }
    public string location { get; set; }
    public string amenities { get; set; }
    public double price_per_night { get; set; }
    public double rating { get; set; }
    public string[] tags { get; set; }
}

// Add sample travel data
var sampleHotels = new List<Hotel>
{
    new Hotel
    {
        id = "1",
        name = "Le Meurice Paris",
        description = "Luxury palace hotel with Michelin-starred dining and views of the Tuileries Garden",
        location = "Paris, France",
        amenities = "Spa, Michelin Restaurant, Concierge, Room Service, Fitness Center",
        price_per_night = 850,
        rating = 4.8,
        tags = new[] { "luxury", "romantic", "historic", "fine-dining", "spa" }
    },
    new Hotel
    {
        id = "2",
        name = "Four Seasons Maui",
        description = "Beachfront resort with world-class spa and family-friendly activities",
        location = "Maui, Hawaii",
        amenities = "Beach Access, Kids Club, Multiple Pools, Spa, Golf Course",
        price_per_night = 695,
        rating = 4.7,
        tags = new[] { "beach", "family-friendly", "resort", "spa", "golf" }
    },
    new Hotel
    {
        id = "3",
        name = "Aman Tokyo",
        description = "Minimalist luxury hotel with panoramic city views and traditional onsen",
        location = "Tokyo, Japan",
        amenities = "Onsen, City Views, Fine Dining, Spa, Business Center",
        price_per_night = 780,
        rating = 4.9,
        tags = new[] { "luxury", "business", "spa", "city", "minimalist" }
    },
    new Hotel
    {
        id = "4",
        name = "Hotel Sacher Vienna",
        description = "Historic hotel home of the original Sachertorte with elegant rooms",
        location = "Vienna, Austria",
        amenities = "Historic Cafe, Concierge, Accessible Rooms, Pet-Friendly",
        price_per_night = 420,
        rating = 4.6,
        tags = new[] { "historic", "accessible", "pet-friendly", "cultural", "cafe" }
    },
    new Hotel
    {
        id = "5",
        name = "Fairmont Whistler",
        description = "Ski-in/ski-out resort with family suites and mountain views",
        location = "Whistler, Canada",
        amenities = "Ski Access, Family Suites, Heated Pool, Kids Programs",
        price_per_night = 380,
        rating = 4.5,
        tags = new[] { "ski", "family-friendly", "mountain", "resort", "accessible" }
    }
};

// Upload hotels to search index
await travelSearchClient.IndexDocumentsAsync(IndexDocumentsBatch.Upload(sampleHotels));
Console.WriteLine($"✅ Uploaded {sampleHotels.Count} hotels to search index");

## Simple In-Memory Implementation

For this C# demo, we'll use a simplified in-memory approach instead of Mem0 (which is Python-based). In production, you would integrate with Azure AI Search's vector capabilities or use a dedicated memory service.

In [8]:
// Simple memory store for demonstration
public class SimpleMemoryStore
{
    private readonly Dictionary<string, List<string>> _userMemories = new();

    public void AddMemory(string userId, string memory)
    {
        if (!_userMemories.ContainsKey(userId))
        {
            _userMemories[userId] = new List<string>();
        }
        _userMemories[userId].Add(memory);
    }

    public List<string> GetMemories(string userId)
    {
        return _userMemories.ContainsKey(userId) 
            ? _userMemories[userId] 
            : new List<string>();
    }

    public List<string> SearchMemories(string userId, string query)
    {
        var memories = GetMemories(userId);
        // Simple keyword search (in production, use semantic search)
        return memories.Where(m => 
            m.Contains(query, StringComparison.OrdinalIgnoreCase)
        ).ToList();
    }
}

var memoryStore = new SimpleMemoryStore();
Console.WriteLine("✅ Memory store initialized");

## Create the Travel Booking Plugin

This plugin provides functions for searching hotels and managing user preferences through our memory store.

In [9]:
public class TravelBookingTools
{
    private readonly SearchClient _searchClient;
    private readonly SimpleMemoryStore _memoryStore;

    public TravelBookingTools(SearchClient searchClient, SimpleMemoryStore memoryStore)
    {
        _searchClient = searchClient;
        _memoryStore = memoryStore;
    }

    [Description("Search for hotels based on criteria like location, amenities, or tags")]
    public async Task<string> SearchHotels(
        [Description("Search query for hotels (location, amenities, etc.)")] string query,
        [Description("Maximum number of results to return")] int maxResults = 3)
    {
        var searchOptions = new SearchOptions
        {
            Size = maxResults,
            IncludeTotalCount = true
        };

        var results = await _searchClient.SearchAsync<Hotel>(query, searchOptions);
        var hotels = new List<object>();

        await foreach (var result in results.Value.GetResultsAsync())
        {
            hotels.Add(new
            {
                name = result.Document.name,
                location = result.Document.location,
                description = result.Document.description,
                price_per_night = result.Document.price_per_night,
                rating = result.Document.rating,
                amenities = result.Document.amenities,
                tags = result.Document.tags
            });
        }

        return JsonConvert.SerializeObject(hotels, Formatting.Indented);
    }

    [Description("Store user travel preferences and important information in memory")]
    public string StoreUserPreference(
        [Description("User identifier")] string userId,
        [Description("User preference or information to remember")] string preference)
    {
        Console.WriteLine($"DEBUG: Storing preference for {userId}: {preference}");
        _memoryStore.AddMemory(userId, preference);
        return $"✅ Stored: {preference}";
    }

    [Description("Get all stored preferences for a user")]
    public string GetUserPreferences(
        [Description("User identifier")] string userId)
    {
        Console.WriteLine($"DEBUG: Getting all preferences for {userId}");
        var memories = _memoryStore.GetMemories(userId);

        if (memories.Count == 0)
        {
            return $"No preferences found for user {userId}";
        }

        return $"User preferences for {userId}:\n- " + string.Join("\n- ", memories);
    }

    [Description("Search user's memories for relevant information")]
    public string SearchMemories(
        [Description("User identifier")] string userId,
        [Description("What to search for (e.g., 'family vacation', 'dietary restrictions')")] string query)
    {
        Console.WriteLine($"DEBUG: Searching memories for {userId} with query: '{query}'");
        var memories = _memoryStore.SearchMemories(userId, query);

        if (memories.Count == 0)
        {
            return $"No memories found for query: {query}";
        }

        return "Relevant memories:\n- " + string.Join("\n- ", memories);
    }
}

Console.WriteLine("✅ TravelBookingTools defined");

## Initialize the AI Agent with Microsoft Agent Framework

Create our travel booking agent with access to the travel booking tools.

In [10]:
// Create and add the travel booking tools
var travelTools = new TravelBookingTools(travelSearchClient, memoryStore);

// Agent instructions
var AGENT_INSTRUCTIONS = @"
You are a personalized travel booking assistant with memory.

WORKFLOW:
1. When a user asks for help, search their memories using SearchMemories with a relevant query
2. Use the memories to personalize your response
3. Store any new preferences they mention using StoreUserPreference
4. When the user is booking a new trip, first retrieve the user's general travel preferences
5. Then use SearchHotels to find suitable options
6. Do not recommend hotels that are over budget

IMPORTANT: For ALL memory operations, use userId='sarah_johnson_123' exactly as written.

Always acknowledge what you found in their memories when responding.
";

// Create the travel agent with Agent Framework
var travelAgent = new AzureOpenAIClient(
    new Uri(azureOpenAIEndpoint),
    new AzureCliCredential())
        .GetChatClient(azureOpenAIDeployment)
        .AsIChatClient()
        .CreateAIAgent(
            name: "TravelBookingAssistant",
            instructions: AGENT_INSTRUCTIONS,
            tools: [
                AIFunctionFactory.Create(travelTools.SearchHotels),
                AIFunctionFactory.Create(travelTools.StoreUserPreference),
                AIFunctionFactory.Create(travelTools.GetUserPreferences),
                AIFunctionFactory.Create(travelTools.SearchMemories)
            ]
        );



## Test the Agent - Store Preferences

In [ ]:
// First interaction - store preferences
var userMessage1 = "Hi! I'm planning trips for this year. I prefer luxury hotels with spa services and I love romantic destinations. My budget is around $500-800 per night.";
Console.WriteLine($"\n👤 User: {userMessage1}\n");
Console.WriteLine("🤖 Agent:\n");

await foreach (var update in travelAgent.RunStreamingAsync(userMessage1, thread))
{
    await Task.Delay(10);
    Console.Write(update);
}

Console.WriteLine("\n");

## Test the Agent - Retrieve and Use Preferences

In [ ]:
// Second interaction - use stored preferences
var userMessage2 = "Can you help me find a hotel for my next trip? I'm thinking somewhere in Europe.";
Console.WriteLine($"\n👤 User: {userMessage2}\n");
Console.WriteLine("🤖 Agent:\n");

await foreach (var update in travelAgent.RunStreamingAsync(userMessage2, thread))
{
    await Task.Delay(10);
    Console.Write(update);
}

Console.WriteLine("\n✅ Conversation completed");

## Summary

This notebook demonstrated:

1. **Memory Integration**: Building an agent that remembers user preferences across conversations
2. **Azure AI Search**: Using Azure AI Search for hotel data and potentially for vector-based memory storage
3. **Semantic Kernel Plugins**: Creating plugins that combine search and memory capabilities
4. **Personalized Responses**: Leveraging stored preferences to provide customized recommendations

### Key Takeaways:

- ✅ **Persistent Memory**: Agents can store and retrieve user-specific information
- ✅ **Context-Aware**: Memory enables personalized, context-aware interactions
- ✅ **Plugin Architecture**: Semantic Kernel plugins provide clean separation of concerns
- ✅ **Scalable Design**: Architecture can scale with Azure AI Search for production use

### Next Steps:

For production implementations:
- Integrate Azure AI Search vector capabilities for semantic memory search
- Add authentication and user management
- Implement memory expiration and cleanup policies
- Add more sophisticated preference learning algorithms
- Consider using Azure Cosmos DB for memory persistence